# MODULE 5 — GroupBy Operations
GroupBy = Split → Apply → Combine
Clean production-ready Pandas examples



In [ ]:
# ============================================================
# MODULE 5 — GROUPBY OPERATIONS (PANDAS PRACTICE FILE)
# ============================================================
# PURPOSE OF THIS FILE:
# This file demonstrates real-world data aggregation, transformation,
# and analysis using Pandas GroupBy operations.
# It simulates a sales dataset and shows how companies analyze revenue,
# performance, and patterns using grouped data.
#
# WHY WE CREATE THIS FILE:
# - To learn how raw data becomes meaningful insights
# - To practice real-world analytics workflows
# - To understand how SQL-like operations work in Pandas
# - To prepare for data analysis / data science tasks
#
# ============================================================
# METHOD CHEAT NOTES (SHORT MEANING)
# groupby   → splits data into groups for analysis
# agg       → creates summary reports (sum, mean, count, etc.)
# transform → returns group-based results for each row
# pivot     → reshapes data into table format (matrix style)
# crosstab  → frequency table between two categorical columns
# isin      → checks membership for filtering data
# axis      → controls direction (0 = rows, 1 = columns)
# p         → probability concept used in sampling distributions
# ============================================================

In [3]:
import pandas as pd
import numpy as np

np.random.seed(42)

df = pd.DataFrame({
    'order_id': range(1, 1001),
    'region': np.random.choice(['North','South','East','West'], 1000),
    'product': np.random.choice(['A','B','C','D'], 1000),
    'sales_rep': np.random.choice(['Alice','Bob','Charlie','Diana'], 1000),
    'revenue': np.random.randint(100, 10000, 1000),
    'discount_pct': np.random.choice([0,5,10,15,20], 1000, p=[0.6,0.2,0.1,0.05,0.05])
})

print(df.shape)
df.head()

(1000, 6)


,order_id,region,product,sales_rep,revenue,discount_pct
0,1,East,B,Charlie,2963,20
1,2,West,C,Diana,1277,10
2,3,North,A,Bob,4860,5
3,4,East,A,Diana,111,0
4,5,East,A,Diana,5352,0


In [5]:
# Safety check
required = ['region','product','sales_rep','revenue','order_id','discount_pct']
missing = [c for c in required if c not in df.columns]
assert not missing, f"Missing columns: {missing}"

In [6]:
# 5.1 Basic GroupBy
df.groupby('region')['revenue'].sum().sort_values(ascending=False)

region_summary = df.groupby('region')['revenue'].agg(['sum','mean','median','count','max'])
region_summary


,sum,mean,median,count,max
region,,,,,
East,1219251,5255.392241,5544.0,232,9966
North,1338661,5188.608527,5264.5,258,9990
South,1126474,4897.713043,5108.0,230,9984
West,1392390,4972.821429,5128.5,280,9973


In [8]:
# 5.2 Multi-column GroupBy
product_region = df.groupby(['region','product'])['revenue'].agg(['sum','count'])
product_region.columns = ['total_revenue','order_count']
product_region['avg_order'] = product_region['total_revenue'] / product_region['order_count']
product_region.sort_values('total_revenue', ascending=False).head(10)


total_revenue  order_count    avg_order
region product                                         
North  B               411450           75  5486.000000
       C               397099           81  4902.456790
West   D               376592           70  5379.885714
South  A               369027           73  5055.164384
East   A               343066           68  5045.088235
       B               340561           62  5492.919355
West   A               340003           70  4857.185714
       B               338155           66  5123.560606
       C               337640           74  4562.702703
South  B               316218           60  5270.300000

In [9]:
# 5.3 Named Aggregation# 5.3 Named Aggregation
kpi = df.groupby('sales_rep').agg(
    total_revenue=('revenue','sum'),
    order_count=('order_id','count'),
    avg_order=('revenue','mean'),
    max_deal=('revenue','max'),
    unique_products=('product','nunique'),
    avg_discount=('discount_pct','mean')
).sort_values('total_revenue', ascending=False)

kpi


,total_revenue,order_count,avg_order,max_deal,unique_products,avg_discount
sales_rep,,,,,,
Diana,1310263,241,5436.775934,9984,4,3.215768
Alice,1289684,261,4941.318008,9936,4,3.563218
Charlie,1241531,240,5173.045833,9985,4,4.229167
Bob,1235298,258,4787.976744,9990,4,3.953488


In [19]:
# 5.4 Custom aggregation# 5.4 Custom aggregation
def discount_rate(x):
    return (x>0).mean()*100 # % of orders with a discount

rep = df.groupby('sales_rep').agg(
    total_revenue=('revenue','sum'),
    discount_rate=('discount_pct', discount_rate),
    revenue_std=('revenue','std')
)

rep
# add standard deviation to measure consistency of sales rep performance

,total_revenue,discount_rate,revenue_std
sales_rep,,,
Alice,1289684,39.846743,2809.576507
Bob,1235298,41.472868,2951.747446
Charlie,1241531,45.000000,2930.528750
Diana,1310263,36.099585,2846.995598


In [11]:
# 5.5 Transform
df['region_total'] = df.groupby('region')['revenue'].transform('sum') # total revenue for each region
df['region_pct'] = (df['revenue']/df['region_total']*100).round(2) # % contribution of each order to region total

def safe_z(x): # safe z-score to handle small groups or zero std
    if len(x) < 2 or x.std() == 0:
        return x - x.mean()
    return (x - x.mean()) / x.std()

df['zscore'] = df.groupby('sales_rep')['revenue'].transform(safe_z).round(2) # z-score of each order's revenue compared to that rep's average
df[['order_id','region','revenue','region_total','region_pct']].head() 



,order_id,region,revenue,region_total,region_pct
0,1,East,2963,1219251,0.24
1,2,West,1277,1392390,0.09
2,3,North,4860,1338661,0.36
3,4,East,111,1219251,0.01
4,5,East,5352,1219251,0.44


In [14]:
# 5.6 Filter groups
rep_sum = df.groupby('sales_rep')['revenue'].sum()
top = rep_sum[rep_sum>50000].index
high_perf = df[df['sales_rep'].isin(top)].copy()


In [15]:
# 5.7 Pivot
pivot = df.pivot_table(values='revenue', index='region', columns='product', aggfunc='sum', fill_value=0)
row_sum = pivot.sum(axis=1).replace(0, 1)
pivot_pct = pivot.div(row_sum, axis=0) * 100

pivot_pct

product,A,B,C,D
region,,,,
East,28.137438,27.931984,23.995305,19.935272
North,21.287242,30.735937,29.663895,18.312926
South,32.759478,28.071487,20.999242,18.169794
West,24.418661,24.285940,24.248953,27.046445


In [18]:
# 5.8 Crosstab
ct = pd.crosstab(df['region'], df['product'], normalize='index')
(ct*100).round(1)

product,A,B,C,D
region,,,,
East,29.3,26.7,23.3,20.7
North,21.7,29.1,31.4,17.8
South,31.7,26.1,21.7,20.4
West,25.0,23.6,26.4,25.0
